In [ ]:
from JointTemporalModel import JointTemporalModel
import torch, math
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

In [ ]:
train = 0
eval = 1
test = 2

In [ ]:
NUM_EPOCHS = 10
ENC_LR = 5e-5
NONENC_LR = 1e-3
WARMUP_EPOCHS = 2
BASE_ENC_MODEL = "roberta-base"

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = JointTemporalModel(base=BASE_ENC_MODEL,
                           num_ner=5, ee_labels=7,
                           heads=4, et_feat_dim=18, ee_feat_dim=16).to(device)

for p in model.enc.parameters(): p.requires_grad = False

optimizer = AdamW([
        {"params": [p for n,p in model.named_parameters() if n.startswith("enc.")], "lr": ENC_LR},
        {"params": [p for n,p in model.named_parameters() if not n.startswith("enc.")], "lr": NONENC_LR},
    ], weight_decay=0.01)

num_train_steps = len(train) * NUM_EPOCHS

sched = get_linear_schedule_with_warmup(optimizer, int(0.05*num_train_steps), num_train_steps)

scaler = torch.amp.GradScaler("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
for epoch in range(NUM_EPOCHS):
    model.train()
    for batch in train_loader:
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k,v in batch.items()}
        with torch.cuda.amp.autocast():
            out = model(**batch)    # only computes losses present in batch
            loss = out["loss"]
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); sched.step()

    # ---- unfreeze after warmup ----
    if epoch == warmup_epochs-1:
        for p in model.enc.parameters(): p.requires_grad = True   # or just top 4 layers
        optimizer = make_optim()  # rebuild optim so encoder has its LR
        sched = get_linear_schedule_with_warmup(optimizer, int(0.06*num_train_steps), num_train_steps)

    # ---- validation ----
    model.eval()
    with torch.no_grad():
        ner_f1, ptr_hits1, ee_macroF1 = evaluate(model, dev_loader, id_maps)
    print(f"ep{epoch}: NER F1={ner_f1:.1f}  PTR@1={ptr_hits1:.1f}  EE mF1={ee_macroF1:.1f}")
    # save best
    save_if_best(...)